# Sepia iNaturalist: anotación manual y modelos

Notebook para revisar con Martin. Solo lee los datos de Drive; **no llama a ningún modelo**.

1. Interfaz de anotación
2. Resultados de los 4 prompts: métricas y matrices de confusión
3. Todas las fotos, una a una
4. Fotos donde al menos un prompt difiere de la anotación manual
5. Fotos donde los 4 prompts coinciden entre sí y difieren de la anotación manual
6. Fotos con notas para preguntar a Martin

En las vistas 4 y 5, `unclear` cuenta siempre como diferencia (en gris en la tabla). En las métricas de la sección 2, `unclear` del modelo es una abstención.

In [ ]:
# --- Preparacion ---
from google.colab import drive, output
drive.mount('/content/drive')
output.enable_custom_widget_manager()

import os
if not os.path.exists('/content/sepia-inaturalist'):     # sesion nueva: clonar el repo
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    !git clone -q https://{token}@github.com/hugozeballos/sepia-inaturalist.git /content/sepia-inaturalist
%cd /content/sepia-inaturalist
!git pull -q
!git log -1 --oneline

import sys
sys.path.insert(0, '/content/sepia-inaturalist/src')
for m in [m for m in sys.modules if m == 'sepia' or m.startswith('sepia.')]:
    del sys.modules[m]                      # usar siempre el codigo recien bajado
import importlib
sys.path_importer_cache.pop('/content/sepia-inaturalist/src', None)   # si la carpeta no existia antes
importlib.invalidate_caches()
import sepia
from sepia.annotate import AnnotationApp
from sepia.disagree import (DisagreementApp, classification_metrics, plot_confusion,
                            count_agreement)

CONFIG = 'configs/pilot_a1.yaml'                   # corrida a mostrar
VERSIONS = ['photo_traits_v14', 'photo_traits_v13', 'photo_traits_v12', 'photo_traits_v8']
ANNOTATOR = 'HZ'

cfg = sepia.load_config(CONFIG)
for v in VERSIONS:
    print(f"{v}: {len(sepia.load_photo_traits(cfg, v))} fotos con resultado")

## 1. Interfaz de anotación

Anotación manual por foto e individuo. **Guarda de verdad** en `annotations/manual_HZ.jsonl` (cada guardado agrega una línea; vale la última por registro + foto + letra). Para mostrarla sin cambiar nada: navegar sin apretar *Save*.

In [ ]:
AnnotationApp(cfg, ANNOTATOR).show()

## 2. Resultados de los 4 prompts

Sobre las mismas fotos para los 4 prompts. Lado, erosión y calidad: solo fotos con 1 individuo anotado.

- **coverage**: fracción de fotos donde el modelo decidió (no `unclear`).
- **accuracy**: aciertos / fotos (`unclear` = error). **accuracy_when_decided**: aciertos cuando decide.
- **precision** de una clase: de las veces que el modelo dijo esa clase, cuántas acertó. **recall**: de las fotos anotadas con esa clase, cuántas encontró. **F1**: combina ambas.
- **macro_f1** y **balanced_accuracy**: promedios por clase, no dependen de cuántas fotos hay de cada clase.

In [ ]:
# Huesos visibles por foto
count_agreement(cfg, VERSIONS);

In [ ]:
# Lado, erosion y calidad: resumen, metricas por clase y matrices
summary, per_class, mats = classification_metrics(cfg, VERSIONS, save=False)

In [ ]:
plot_confusion(mats, 'side');           # fila 'unclear' = anotado unclear (fuera de las metricas)
plot_confusion(mats, 'erosion');
plot_confusion(mats, 'photo_quality');

## 3. Todas las fotos, una a una

Foto grande, miniaturas del registro (clic = ver en grande), tabla con los 4 prompts y la anotación manual, evidencia de cada prompt y resumen del registro. *Field* filtra por criterio; *Go to ID* salta a un registro.

In [ ]:
DisagreementApp(cfg, VERSIONS, ANNOTATOR, view='all', read_only=True).show()

## 4. Al menos un prompt difiere de la anotación manual

En cualquier criterio (o en el que se elija en *Field*). Rojo = el prompt decidió otra cosa; gris = el prompt dijo `unclear`.

In [ ]:
DisagreementApp(cfg, VERSIONS, ANNOTATOR, view='any_model_differs', read_only=True).show()

## 5. Los 4 prompts coinciden entre sí y difieren de la anotación manual

Candidatas más fuertes a error de anotación, o a un criterio distinto entre la persona y los prompts. Incluye los casos donde los 4 dicen `unclear` y la persona decidió, y donde la persona dijo `unclear` y los 4 decidieron lo mismo.

In [ ]:
DisagreementApp(cfg, VERSIONS, ANNOTATOR, view='all_models_agree_differ', read_only=True).show()

## 6. Preguntas para Martin

Fotos donde alguna nota de la anotación menciona a Martin (martin, Martín, marthin). La nota aparece destacada arriba.

In [ ]:
DisagreementApp(cfg, VERSIONS, ANNOTATOR, view='ask_martin', read_only=True).show()